In [0]:
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.gold_claim_summary")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.gold_fraud_signals")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_claims")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_payments")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_providers")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_patients")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_rules")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_audit_logs")
# spark.sql("DROP TABLE IF EXISTS healthcare_fraud_demo.silver_claim_lines")
# print("All silver tables dropped")

In [0]:
# dbutils.fs.rm("/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/silver_claims", True)
# dbutils.fs.rm("/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/silver_payments", True)
# dbutils.fs.rm("/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/silver_claim_lines", True)
# dbutils.fs.rm("/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/gold_claim_summary", True)
# dbutils.fs.rm("/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/gold_fraud_signals", True)
# print("All checkpoints dropped")


In [0]:
# MUST be first line
spark.sql("USE CATALOG health_care_2026")

db = "healthcare_fraud_demo"

# In Unity Catalog use SCHEMA not DATABASE
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

base_path = "/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud"

checkpoint = f"{base_path}/checkpoints"
schema_loc = f"{base_path}/schema"

print(checkpoint)
print(schema_loc)

In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date, get_json_object, lit
bronze_claims = spark.table(f"{db}.bronze_claims")

claims_clean = (
    bronze_claims
    .withColumn("claim_id", col("claim_id").cast("int"))
    .withColumn("provider_id", col("provider_id").cast("int"))
    .withColumn("patient_id", col("patient_id").cast("int"))
    .withColumn("claim_amount", col("claim_amount").cast("double"))

    # SAFE MULTI FORMAT DATE
    .withColumn(
        "claim_date",
        coalesce(
            try_to_date(col("claim_date"), "dd-MM-yyyy"),
            try_to_date(col("claim_date"), "yyyy-MM-dd"),
            try_to_date(get_json_object("_rescued_data","$.claim_date"), "dd-MM-yyyy"),
            try_to_date(get_json_object("_rescued_data","$.claim_date"), "yyyy-MM-dd")
        )
    )

    .filter(col("claim_id").isNotNull())
    .dropDuplicates(["claim_id"])
    .withColumn("is_active", lit(True))
)

claims_clean.write.format("delta").saveAsTable(
    f"{db}.silver_claims"
)

print("Silver claims created")

In [0]:
bronze_providers = spark.table(f"{db}.bronze_providers")

silver_providers = (
    bronze_providers
    .withColumn("provider_id", col("provider_id").cast("int"))
    .withColumn("name", col("name").cast("string"))
    .withColumn("specialty", col("specialty").cast("string"))
    .withColumn("risk_score", col("risk_score").cast("double"))
    .dropDuplicates(["provider_id"])
)

silver_providers.write.format("delta").saveAsTable(
    f"{db}.silver_providers"
)

print("Silver providers created")


In [0]:
bronze_patients = spark.table(f"{db}.bronze_patients")

silver_patients = (
    bronze_patients
    .withColumn("patient_id", col("patient_id").cast("int"))
    .withColumn("age", col("age").cast("int"))
    .withColumn("gender", col("gender").cast("string"))
    .dropDuplicates(["patient_id"])
)

silver_patients.write.format("delta").saveAsTable(
    f"{db}.silver_patients"
)

print("Silver patients created")


In [0]:
bronze_payments = spark.table(f"{db}.bronze_payments")

silver_payments = (
    bronze_payments
    .withColumn("payment_id", col("payment_id").cast("int"))
    .withColumn("claim_id", col("claim_id").cast("int"))
    .withColumn("paid_amount", col("paid_amount").cast("double"))
    .dropDuplicates(["payment_id"])
)

silver_payments.write.format("delta").saveAsTable(
    f"{db}.silver_payments"
)

print("Silver payments created")


In [0]:
%sql
SHOW TABLES IN healthcare_fraud_demo;
